# AI-Based Phishing Website Detection System
### End-to-End Machine Learning Pipeline, Feature Engineering & Explainable Risk Assessment

**Author:** A Karthick ([@karthick0328-design](https://github.com/karthick0328-design))

---

## 1. Overview
Phishing remains one of the primary threat vectors in cybersecurity. Malicious actors continuously devise evasive URLs designed to mimic legitimate institutions. Traditional blacklist-based detection suffers from zero-day lag. 

This notebook demonstrates a **lexical and structural machine learning pipeline** that evaluates URLs without requiring active webpage rendering, ensuring zero payload execution danger while achieving high recall and precision.

## 2. Environment Setup & Imports

In [ ]:
import re
import math
import urllib.parse
from collections import Counter
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

print("All dependencies imported successfully!")

## 3. Feature Engineering Pipeline
We extract 15 lexical and structural features from each URL, capturing:
- **Lexical indicators:** Length, entropy, dot count, hyphen count, digit ratio
- **Structural markers:** Subdomain depth, path depth, query parameter count
- **Security anomalies:** Presence of raw IPv4 address, suspicious security keywords (`verify`, `login`, `banking`), and URL shortening services.

In [ ]:
SUSPICIOUS_KEYWORDS = [
    "login", "verify", "account", "update", "banking", "secure",
    "confirm", "password", "credential", "wallet", "signin",
    "authenticate", "support", "service", "billing", "invoice"
]

KNOWN_SHORTENERS = {
    "bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly",
    "is.gd", "buff.ly", "adf.ly", "tiny.cc", "rb.gy"
}

FEATURE_NAMES = [
    "url_length",
    "hostname_length",
    "path_length",
    "dot_count",
    "hyphen_count",
    "at_symbol_count",
    "question_count",
    "digit_count",
    "digit_ratio",
    "subdomain_count",
    "shannon_entropy",
    "is_ip_address",
    "is_shortener",
    "suspicious_keyword_count",
    "uses_https"
]

def calculate_entropy(text: str) -> float:
    if not text:
        return 0.0
    counts = Counter(text)
    length = len(text)
    return float(-sum((c / length) * math.log2(c / length) for c in counts.values()))

def extract_features(raw_url: str) -> Dict[str, Any]:
    url = raw_url.strip()
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", url):
        url = "https://" + url
    
    parsed = urllib.parse.urlsplit(url)
    hostname = parsed.hostname or ""
    path = parsed.path or ""
    
    url_len = len(url)
    digit_c = sum(c.isdigit() for c in url)
    subdomains = max(0, len(hostname.split(".")) - 2) if hostname else 0
    is_ip = 1 if re.match(r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$", hostname) else 0
    is_short = 1 if hostname.lower() in KNOWN_SHORTENERS else 0
    kw_count = sum(1 for kw in SUSPICIOUS_KEYWORDS if kw in url.lower())
    https_flag = 1 if parsed.scheme.lower() == "https" else 0
    
    feat_vector = [
        float(url_len),
        float(len(hostname)),
        float(len(path)),
        float(url.count(".")),
        float(url.count("-")),
        float(url.count("@")),
        float(url.count("?")),
        float(digit_c),
        float(digit_c / url_len) if url_len > 0 else 0.0,
        float(subdomains),
        calculate_entropy(url),
        float(is_ip),
        float(is_short),
        float(kw_count),
        float(https_flag)
    ]
    return dict(zip(FEATURE_NAMES, feat_vector))

## 4. Dataset Loading & Feature Extraction

In [ ]:
# Load dataset from local path or fallback to sample
import os
dataset_candidates = ["../data/dataset.csv", "dataset.csv", "data/dataset.csv"]
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), None)

if dataset_path:
    df = pd.read_csv(dataset_path)
    print(f"Loaded dataset from {dataset_path} with {len(df):,} records.")
else:
    print("Using embedded benchmark dataset...")
    sample_data = [
        ("https://www.google.com", 0), ("https://github.com/explore", 0),
        ("https://en.wikipedia.org/wiki/Phishing", 0), ("https://stackoverflow.com", 0),
        ("http://192.168.1.1/admin/login.php", 1), ("http://paypal-verification-update-security.com", 1),
        ("http://apple-secure-account-login.cc/verify", 1), ("http://bit.ly/bank-security-claim", 1)
    ] * 200
    df = pd.DataFrame(sample_data, columns=["url", "label"])

print("Class Distribution:")
print(df["label"].value_counts())

# Extract features
X_rows = []
y_rows = []
for _, row in df.iterrows():
    try:
        feats = extract_features(str(row["url"]))
        X_rows.append(list(feats.values()))
        y_rows.append(int(row["label"]))
    except Exception:
        continue

X = np.array(X_rows, dtype=np.float32)
y = np.array(y_rows, dtype=np.int32)
print(f"Extracted feature matrix shape: {X.shape}")

## 5. Model Training & Comparative Benchmarking
We evaluate 4 representative classifiers:
1. **Logistic Regression** (Linear baseline with Standard Scaling)
2. **Decision Tree Classifier** (Interpretable non-linear tree)
3. **Random Forest Classifier** (Bagging ensemble for variance reduction)
4. **Gradient Boosting Classifier** (Boosting ensemble for sequential error correction)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=16, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
fitted_models = {}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else y_pred
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, proba)
    
    fitted_models[name] = clf
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(auc, 4)
    })

benchmark_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
benchmark_df

## 6. Confusion Matrix & ROC Curves

In [ ]:
best_model_name = benchmark_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
print(f"Selected Champion Model: {best_model_name}")

y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Plot
cax = ax[0].matshow(cm, cmap="Blues", alpha=0.8)
fig.colorbar(cax, ax=ax[0])
for (i, j), val in np.ndenumerate(cm):
    ax[0].text(j, i, f"{val:,}", ha="center", va="center", fontsize=14, fontweight="bold")
ax[0].set_xticklabels(["", "Legitimate (0)", "Phishing (1)"])
ax[0].set_yticklabels(["", "Legitimate (0)", "Phishing (1)"])
ax[0].set_xlabel("Predicted Label", fontweight="bold")
ax[0].set_ylabel("True Label", fontweight="bold")
ax[0].set_title(f"Confusion Matrix ({best_model_name})", pad=15, fontweight="bold")

# ROC Curve
for name, clf in fitted_models.items():
    if hasattr(clf, "predict_proba"):
        p = clf.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, p)
        ax[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")

ax[1].plot([0, 1], [0, 1], "k--", alpha=0.6)
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].set_title("ROC Curves Comparison", fontweight="bold")
ax[1].legend(loc="lower right")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis
Understanding what features drive the predictions provides transparency and explainability.

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    feat_df = pd.DataFrame({
        "Feature": FEATURE_NAMES,
        "Importance": importances
    }).sort_values(by="Importance", ascending=True)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feat_df["Feature"], feat_df["Importance"], color="#2563eb")
    plt.xlabel("Relative Importance", fontweight="bold")
    plt.title(f"Feature Importance Ranking ({best_model_name})", fontweight="bold")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Real-time Inference & Explainability Test

In [ ]:
test_samples = [
    "https://www.google.com",
    "https://github.com/karthick0328-design",
    "http://192.168.0.1/login.php?verify=account",
    "http://secure-paypal-billing-update-verify.cc/account-login"
]

for u in test_samples:
    f_dict = extract_features(u)
    vec = np.array([list(f_dict.values())], dtype=np.float32)
    prob = float(best_model.predict_proba(vec)[0][1])
    pred_label = "Phishing" if prob >= 0.5 else "Legitimate"
    print(f"URL: {u}")
    print(f"  -> Verdict: {pred_label} (Phishing Probability: {prob:.2%})")
    print("-" * 60)